# Fraud detection playground

This notebook lets you poke at the experiment behind the article without installing anything. It runs in Google Colab, a free notebook service from Google.

**How to use it:** each gray box is a small piece of Python code. Click inside one and press **Shift+Enter** to run it; the result appears underneath. Run the boxes in order, top to bottom. If Colab warns that the notebook was not written by Google, click *Run anyway*.

Nothing you do here changes the project online. If something looks wrong, use *Runtime > Restart session* and run the boxes from the top again. To keep your changes, use *File > Save a copy in Drive*.

Lines that start with `#` are notes for you; Python skips them.

In [1]:
# Setup. In Colab this downloads the project (about 5 MB) the first time.
# On your own computer, run the notebook from the project folder and this box only loads the data.
import os, sys, json, subprocess, textwrap
from collections import Counter

if "google.colab" in sys.modules:
    folder = "/content/llm-fraud-classification"
    if not os.path.exists(folder):
        download = subprocess.run(["git", "clone", "--depth", "1",
                                   "https://github.com/Fortuenti-Consulting/llm-fraud-classification.git", folder])
        if download.returncode != 0:
            raise SystemExit("The download failed. Check your internet connection and run this box again.")
    os.chdir(folder)
sys.path.insert(0, ".")

# Load the 5,000 made-up payments. Each one has raw fields (what a bank system stores)
# and a short text description (what the AI model reads).
with open("synthetic_fraud_dataset.json") as f:
    dataset = json.load(f)
transactions = {r["raw_ml_features"]["transaction_id"]: r for r in dataset}
print(f"Loaded {len(dataset):,} payments")
print(Counter(r["raw_ml_features"]["fraud_category"] for r in dataset))

Loaded 5,000 payments
Counter({'Legitimate': 4607, 'Stolen_Card': 105, 'Carding': 103, 'Synthetic_Identity': 97, 'BIN_Testing': 88})


## 1. One payment, two ways

A bank's systems store a payment as dozens of fields. The language models in this study never see those fields directly: they read a short description written from them. Below is the same payment both ways.

Try other payments: replace the ID between the quote marks in `TRANSACTION_ID` with one of these, keep the quote marks, and run the box again:
`tx_124a253353` (legitimate), `tx_9bf5f27eda` (carding), `tx_92c3158354` (stolen card), `tx_b992025a28` (synthetic identity).

In [2]:
TRANSACTION_ID = "tx_144223243b"   # a BIN testing attack

record = transactions[TRANSACTION_ID]
raw = record["raw_ml_features"]

print("TRUE LABEL:", raw["fraud_category"])
print("\nA few of the raw fields:")
for field in ["amount", "cvv_result", "auth_response_code", "velocity_pan_count_1m_ip",
              "account_age_days", "email_domain", "billing_zip", "shipping_zip", "session_duration_ms"]:
    print(f"  {field:28} {raw[field]}")

print("\nWhat the AI model reads:")
for part, sentence in record["llm_reasoning_prompt"].items():
    print(textwrap.fill(f"- {sentence}", 100, subsequent_indent="  "))

TRUE LABEL: BIN_Testing

A few of the raw fields:
  amount                       1.0
  cvv_result                   Mismatch
  auth_response_code           N7 - CVV Mismatch
  velocity_pan_count_1m_ip     33
  account_age_days             0
  email_domain                 guerrillamail.com
  billing_zip                  00000
  shipping_zip                 00000
  session_duration_ms          561

What the AI model reads:
- A very low-value micro-purchase of USD 1.00.
- Authorization failed due to a CVV Mismatch (Code N7 - CVV Mismatch).
- Extreme velocity anomaly: This IP has attempted 33 distinct card numbers in the last 60 seconds,
  triggering massive CVV failures on BIN 535539.
- This is a brand-new guest checkout session with zero historical platform trust.
- No physical delivery address provided (digital/pre-auth transaction).
- Severe bot signature: Checkout completed in an impossible 561 milliseconds. Fields were
  programmatically injected bypassing standard keyboard events. C

## 2. The rule engine

Before trying AI: what can plain if-then rules do? `card_fraud_analyzer.py` has five rules, checked in order. In words:

1. More than 10 different cards in a minute **and** the security code (CVV) failed: **BIN testing**
2. More than 5 cards in a minute, the CVV matched **and** the bank declined: **carding**
3. Synthetic-identity score above 0.80, or a throwaway email on an account younger than 14 days: **synthetic identity**
4. Shipping ZIP different from billing ZIP on an account older than 30 days: **stolen card**
5. Anything else: **legitimate**

The box below runs these rules on two datasets: the original one, and the *shifted* records of a second dataset, where each payment was changed just enough to slip past one rule (a bot that slows down to 6-10 cards a minute, a real customer sending a gift to another address). The shifted set was built after the rules were written, the way new attacks show up after a fraud team writes its rules. Because each shifted record was built to beat a rule, the rules are expected to fail there; the interesting question is how the AI models do on the same records.

In [3]:
import card_fraud_analyzer as rules

with open("shifted_fraud_dataset.json") as f:
    shifted = [r for r in json.load(f) if r["raw_ml_features"]["variant"] == "shifted"]

def scorecard(classify, title):
    """Prints how many payments of each type a classify() function gets right."""
    print(title)
    for name, data in [("original data", dataset), ("shifted records", shifted)]:
        right, total = Counter(), Counter()
        for r in data:
            truth = r["raw_ml_features"]["fraud_category"]
            total[truth] += 1
            right[truth] += classify(rules.extract_features(r)) == truth
        print(f"  {name}:")
        for label in ["Legitimate", "BIN_Testing", "Carding", "Stolen_Card", "Synthetic_Identity"]:
            print(f"    {label:20} {right[label]:5,} of {total[label]:5,} labeled correctly ({right[label] / total[label]:.0%})")

scorecard(rules.rule_based_fallback_classifier, "Frozen rules from card_fraud_analyzer.py")

Frozen rules from card_fraud_analyzer.py
  original data:
    Legitimate           4,607 of 4,607 labeled correctly (100%)
    BIN_Testing             88 of    88 labeled correctly (100%)
    Carding                 99 of   103 labeled correctly (96%)
    Stolen_Card             97 of   105 labeled correctly (92%)
    Synthetic_Identity      86 of    97 labeled correctly (89%)
  shifted records:
    Legitimate               0 of 2,080 labeled correctly (0%)
    BIN_Testing              0 of    44 labeled correctly (0%)
    Carding                  0 of    53 labeled correctly (0%)
    Stolen_Card              0 of    43 labeled correctly (0%)
    Synthetic_Identity       0 of    54 labeled correctly (0%)


Near-perfect on the data the rules were written for, and nothing right on the shifted records. For Legitimate, 0 right means all 2,080 of those real customers would have been blocked as fraud: rule 4 reads every gift sent to another address as a stolen card.

## 3. Your turn: change a limit

Below is a copy of the same five rules with their limits pulled out as settings. Change a number, run the box, and compare.

Something to try: the shifted BIN testing bots send 6 to 10 cards a minute. Set `BIN_TESTING_MIN_CARDS = 5` and run the box. You should see BIN testing on the shifted records go from 0 to 44 of 44, with nothing else changing. Patching a rule is easy once you know about the attack; the hard part is noticing it first. Then try loosening another limit and watch which other category starts to suffer.

In [4]:
BIN_TESTING_MIN_CARDS = 10     # rule 1: more than this many cards a minute
CARDING_MIN_CARDS = 5           # rule 2: more than this many cards a minute
SYNTHETIC_SCORE_LIMIT = 0.80    # rule 3: synthetic-identity score above this
NEW_ACCOUNT_DAYS = 14           # rule 3: "new account" means younger than this
STOLEN_MIN_ACCOUNT_DAYS = 30    # rule 4: account older than this

def my_rules(f):
    if f["card_velocity_60s"] > BIN_TESTING_MIN_CARDS and not f["cvv_match"]:
        return "BIN_Testing"
    if f["card_velocity_60s"] > CARDING_MIN_CARDS and f["cvv_match"] and f["declined_code"]:
        return "Carding"
    if f["synthetic_identity_score"] > SYNTHETIC_SCORE_LIMIT or (f["is_disposable_email"] and f["account_age_days"] < NEW_ACCOUNT_DAYS):
        return "Synthetic_Identity"
    if not f["billing_shipping_zip_match"] and f["account_age_days"] > STOLEN_MIN_ACCOUNT_DAYS:
        return "Stolen_Card"
    return "Legitimate"

scorecard(my_rules, "Your rules")

Your rules
  original data:
    Legitimate           4,607 of 4,607 labeled correctly (100%)
    BIN_Testing             88 of    88 labeled correctly (100%)
    Carding                 99 of   103 labeled correctly (96%)
    Stolen_Card             97 of   105 labeled correctly (92%)
    Synthetic_Identity      86 of    97 labeled correctly (89%)
  shifted records:
    Legitimate               0 of 2,080 labeled correctly (0%)
    BIN_Testing              0 of    44 labeled correctly (0%)
    Carding                  0 of    53 labeled correctly (0%)
    Stolen_Card              0 of    43 labeled correctly (0%)
    Synthetic_Identity       0 of    54 labeled correctly (0%)


## 4. Read a model's reasoning

In the chain-of-thought runs the model was asked to think out loud before answering. Its notes (the *scratchpad*) were saved, which lets you see *why* it got something wrong.

Here GPT-4o-mini looks at the BIN testing attack from section 1. Read its notes and spot the moment it goes off track. The prompt for this run listed the label names but not their definitions, so the model fell back on its own idea of what "carding" means.

In [5]:
import csv

with open("llm_eval_log_cot_openai.csv") as f:
    cot_rows = {row["transaction_id"]: row for row in csv.DictReader(f)}

row = cot_rows["tx_144223243b"]
print("TRUE LABEL:", row["true_label"], "   MODEL SAID:", row["predicted_label"])
print()
for paragraph in row["cot_output"].split("\n"):
    print(textwrap.fill(paragraph, 100))

TRUE LABEL: BIN_Testing    MODEL SAID: Carding

<scratchpad>
1. Cryptographic Signals: The transaction failed with a CVV Mismatch (Code N7 - CVV Mismatch).
2. Velocity Profile: The narrative indicates an extreme velocity anomaly, with 33 distinct card
numbers attempted in the last 60 seconds, specifically targeting BIN 535539.
3. Identity Risk: The identity profile shows a brand-new guest checkout session with zero historical
platform trust, which raises concerns about the legitimacy of the identity.
4. Logistics Risk: There is no physical delivery address provided, indicating this is a digital/pre-
auth transaction, which is common in fraudulent activities.

Based on the analysis, the transaction shows clear signs of being part of a carding operation, given
the failed CVV, extreme velocity, and lack of a legitimate identity or logistics context.
</scratchpad>

<classification>Carding</classification>


## 5. All results in one table

Accuracy on the same 50 payments, average time per answer and total cost of the 50 questions, for each prompt style and model. Full discussion in the README.

In [6]:
import pandas as pd

with open("performance_grid_analysis.json") as f:
    grid = json.load(f)

table = pd.DataFrame([
    {"prompt style": {"zero-shot": "zero-shot", "few-shot": "few-shot", "cot": "chain-of-thought"}[style], "model": {"openai": "GPT-4o-mini", "anthropic": "Claude Haiku 4.5"}[model],
     "accuracy": f"{cell['accuracy']:.0f}%", "avg time": f"{cell['latency'] / 1000:.1f} s", "cost of 50": f"${cell['cost']:.4f}"}
    for style, models in grid.items() for model, cell in models.items()
])
print(table.to_string(index=False))

    prompt style            model accuracy avg time cost of 50
       zero-shot      GPT-4o-mini      74%    0.6 s    $0.0025
       zero-shot Claude Haiku 4.5      74%    0.6 s    $0.0206
        few-shot      GPT-4o-mini      90%    0.5 s    $0.0040
        few-shot Claude Haiku 4.5      92%    0.7 s    $0.0321
chain-of-thought      GPT-4o-mini      56%    2.0 s    $0.0092
chain-of-thought Claude Haiku 4.5      74%    6.5 s    $0.1649


## 6. Optional: ask the model yourself

This needs your own OpenAI API key (create one at platform.openai.com; you pay OpenAI directly, and one question costs well under one cent).

Set `RUN_LIVE_CALL = True` and run the box. It asks for the key in a hidden field and does not save it in the notebook: paste it once (you won't see the characters) and press Enter. Never type a key into a notebook cell or share it.

In Colab the box first installs two small libraries. If Colab then asks you to restart, click *Restart session* and run all the boxes again from the top.

In [7]:
RUN_LIVE_CALL = False
TRANSACTION_ID = "tx_144223243b"

if RUN_LIVE_CALL:
    import getpass, importlib.util
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")
    os.environ.setdefault("ANTHROPIC_API_KEY", "not-used")  # the script expects both; only OpenAI is called here
    if "google.colab" in sys.modules:  # on your own computer, use pip install -r requirements.txt instead
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openai==3.22.0", "anthropic==1.9.0",
                        "httpx2==2.13.1", "python-dotenv"], check=True)
    # Note: on your own computer, a key in the project's .env file takes priority over the one typed here

    # Reuse the exact few-shot prompt and API call from the experiment
    spec = importlib.util.spec_from_file_location("few_shot", "classify_llm_few-shot.py")
    few_shot = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(few_shot)

    record = transactions[TRANSACTION_ID]
    prompt = few_shot.build_few_shot_prompt(record["llm_reasoning_prompt"])
    label, _, _, cost, ms = few_shot.classify_with_llm(prompt, "OPENAI")
    print("TRUE LABEL: ", record["raw_ml_features"]["fraud_category"])
    print("MODEL SAID: ", label)
    print(f"Took {ms:.0f} ms and cost ${cost:.6f}")
else:
    print("Live call skipped. Set RUN_LIVE_CALL = True above to try it.")

Live call skipped. Set RUN_LIVE_CALL = True above to try it.


## Where to go next

- Change the instructions the model gets in `classify_llm_few-shot.py` (the text inside `build_few_shot_prompt`), then rerun that script on your computer.
- Read `Engineer_Report.md` for every reasoning mistake the models made, grouped by cause.
- The README lists more ideas and explains every file.

Built by Ricardo Iguaran at [Fortuenti Consulting](https://fortuenti.co). The README's About section says how to reach us.